In [0]:
SELECT
    *
FROM
    parquet.`/Volumes/workspace/futbol/futbol_landing`
ORDER BY
    date DESC
LIMIT 5;

SELECT
    date,
    name,
    league,
    match_,
    link
FROM
    parquet.`/Volumes/workspace/futbol/futbol_landing`
    LATERAL VIEW EXPLODE(match) m AS match_
ORDER BY
    date DESC
LIMIT 10;

SELECT
    date,
    name,
    league,
    match_.awayTeam.name AS away_team,
    match_.homeTeam.name AS home_team,
    match_.location.name AS stadium,
    cast(match_.startDate as TIMESTAMP) AS start_date,
    cast(match_.endDate as TIMESTAMP) AS end_date,
    -- sub_event,
    CAST(sub_event.startDate AS TIMESTAMP) AS event_time,
    sub_event.attendee.name AS event_team,
    sub_event.name AS description,
    CASE
        WHEN sub_event.name IS NULL THEN ""
        WHEN sub_event.name LIKE 'Gol%' THEN 'GOL'
        WHEN sub_event.name LIKE 'Amonest%' THEN 'TARJETA AMARILLA'
        WHEN sub_event.name LIKE 'Expulsi%' THEN 'TARJETA ROJA'
        WHEN sub_event.name LIKE 'Sale%' THEN 'CAMBIO'
        ELSE 'OTHER'
    END as event_type,
    team.name AS team_name,
    team_players.name AS player_name,
    team_players.roleName AS player_position,
    link
FROM
    parquet.`/Volumes/workspace/futbol/futbol_landing`
    LATERAL VIEW EXPLODE(match) m AS match_
    LATERAL VIEW EXPLODE(match_.subEvent) se AS sub_event
    LATERAL VIEW EXPLODE(match[1].`@graph`) t AS team
    LATERAL VIEW EXPLODE(team.athlete) p AS team_players
ORDER BY
    date DESC
LIMIT 10;

SELECT
  REPLACE(substring_index(match[0].url, '-', -1), "/", "") AS match_id,
  COUNT(*) AS count
FROM
  futbol.bronze_matchs
GROUP BY
  match_id
HAVING
  COUNT(*) > 1
ORDER BY
  count DESC
LIMIT 10
